In [0]:
%pip install yfinance
import yfinance as yf

In [0]:
# data = yf.download("AAPL", period="1y")
# print(data.head())

# multi_data = yf.download(["AAPL", "MSFT", "GOOG"], start="2023-01-01")
# multi_data.display()

import yfinance as yf
#  EXAMPLE FOR HOW TO USE THIS API
# Create a Ticker object
ticker = yf.Ticker("AAPL")

# Get stock info (metadata, market cap, P/E ratio, etc.)
print(ticker.info)

# Get historical data for a specific period
hist = ticker.history(period="1mo")

# # Get financial records
# print(ticker.financials)      # Income Statement
# print(ticker.balance_sheet)   # Balance Sheet
# print(ticker.cashflow)        # Cash Flow Statement

# # Get corporate actions
# print(ticker.dividends)       # Dividend history
# print(ticker.splits)          # Stock splits

In [0]:
TICKERS = {
    "^NSEI":         "Nifty50",
    "^BSESN":        "Sensex",
    "^NSEBANK":      "NiftyBank",
    "NIFTYENERGY.NS":"NiftyEnergy",
    "^CNXAUTO":      "NiftyAuto",
    "^CNXIT":        "NiftyIT",
    "HAL.NS":        "HAL",
    "INDIGO.NS":     "IndiGo",
    "ASIANPAINT.NS": "AsianPaints",
    "ONGC.NS":       "ONGC",
    "BZ=F":          "BrentCrude",
    "INR=X":         "USDINR",
    "GC=F":          "Gold",
    "^INDIAVIX":     "IndiaVIX"
}

In [0]:
# rading day from 2023-Oct-01 to 2025-Mar-31
# Missing tickers = ingestion failure


In [0]:
START_DATE = "2023-10-01"
END_DATE   = "2025-03-31"


In [0]:
from pyspark.sql.functions import lit, current_timestamp
from datetime import datetime

all_dfs = []

for ticker, name in TICKERS.items():
    try:
        df = yf.download(ticker, start=START_DATE, end=END_DATE, interval="1d", auto_adjust=True)
       

        if df.empty:
           print(f"⚠️ Skipping {ticker} — no data")
           continue
        # explicitly defining columns
        # df.columns = ['trade_date', 'open', 'high', 'low', 'close', 'volume']
        # FAILED NIFTYENERGY.NS: Length mismatch: Expected axis has 7 elements, new values have 6 elements
        cols = df.columns.tolist()
        cols[0] = 'trade_date'
        # df.columns = [c.lower().replace(' ', '_') for c in cols]
        
        df.columns = [
        c[0].lower().replace(' ', '_') if isinstance(c, tuple)
        else c.lower().replace(' ', '_')
        for c in df.columns
        ]
        df = df.reset_index()

        # for data lineage and traceblility metadata
        df['source_ticker']    = ticker
        df['ticker_name']      = name
        df['ingestion_timestamp'] = datetime.now()
        df['source_file']      = 'yfinance_api'
        all_dfs.append(df)
        print(f"✅ {ticker} ({name}): {len(df)} rows")
    except Exception as e:
        print(f"❌ FAILED {ticker}: {e}")

# Cell 4: Convert to Spark and write to Bronze
combined_df = pd.concat(all_dfs, ignore_index=True)
spark_df = spark.createDataFrame(combined_df)

In [0]:
spark_df.display()

In [0]:
# combined_df = []

# for ticker,name in TICKERS.items():
#     print(f"Processing {ticker}...")
#     df = yf.download(ticker, start=START_DATE, end=END_DATE, interval="1d", auto_adjust=True)
#     df = df.reset_index()
#     print(df.head())
#     combined_df.append(df)


In [0]:
# print(combined_df[0].columns)

# # MultiIndex([(  'Date',      ''),
# #             ( 'Close', '^NSEI'),
# #             (  'High', '^NSEI'),
# #             (   'Low', '^NSEI'),
# #             (  'Open', '^NSEI'),
# #             ('Volume', '^NSEI')],
# #            names=['Price', 'Ticker'])

# # returns a MultiIndex column structure
# # Flattening the MultiIndex → Single-level columns
# combined_df[0].columns = [col[0].lower() for col in combined_df[0].columns]

# print(combined_df[0].columns)




In [0]:
# combined_df[1].display()
#  list to a single df
# import pandas as pd
# df_total = pd.concat(combined_df,ignore_index=True)
# df = spark.createDataFrame(df_total)


# df.display()
# Each ticker produces different column labels, because:

# Columns are not just 'Close'
# They are ('Close', ticker) → different for every ticker

# So Pandas treats them as different columns
